# 12 · Compact information for judging existing histogram answers

**Research question:** can a small query-independent map of source-density heterogeneity recover some of the
reliability-prediction benefit of full source histograms, within a fixed serialized storage budget?

This is one exploratory mechanism prototype. ARM exports and the existing histogram answer formula remain unchanged.
No learned merge policy, answer correction, classifier search or new calibration algorithm is introduced. It does not
run exact fallback or claim an end-to-end speedup. True counts are used offline for training/evaluation only.

## Execute on Databricks

1. Use your personal Git folder and an ML cluster. Run the install cell, then the configuration cell to create widgets.
2. Normally **leave `run_dir` blank**: the latest completed 08d `source-information` run under your personal `output_root`
   is selected by modification time only and printed. To reproduce a particular run, set its complete directory explicitly.
   The source 08d run supplies matching histogram/query exports, hashes and saved query splits; its trained models are not needed.
3. Keep defaults for a first run: seed 0, nearest saved rate to 0.5, query split, original LightGBM budget of 4000 trees.
   All eligible queries at the selected rate are used for quality evaluation; `timing_queries=30` limits only cost sampling.
4. Run All. Every attempt writes a new `compact-quality-...` directory. Inputs and model/payload identities are preserved.

For a larger, predeclared comparison use `seeds=0,1,2`, `splits=query,cell` and optionally `merge_rates=all`.
That can fit up to 36 models and evaluate many repeated query/histogram cases, so first complete the default run.
No personal user path is hard-coded. Set `data_dir`/`histogram_dir` only if the 08d inputs were relocated; hashes must match.

In [ ]:
%pip install lightgbm==4.6.0

In [ ]:
import hashlib
import importlib
import importlib.metadata
import json
import math
import os
import platform
import shutil
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4
from zipfile import ZipFile, ZIP_DEFLATED

import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy.ndimage import uniform_filter
from sklearn.metrics import roc_auc_score


def notebook_defaults():
    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        root = context.notebookPath().get().split('/ml/notebooks/')[0]
        repo = root if root.startswith('/Workspace/') else '/Workspace' + root
        output = '/Workspace/Users/' + context.userName().get() + '/synopsislake-results'
    except (NameError, AttributeError):
        repo, output = str(Path.cwd()), str(Path(tempfile.gettempdir()) / 'synopsislake-results')
    return dict(repo_root=repo, output_root=output, run_dir='', data_dir='', histogram_dir='',
                seeds='0', merge_rates='0.5', splits='query', max_trees='4000', num_threads='2',
                timing_queries='30', timing_repeats='3', random_seed='20261113')


DEFAULTS = notebook_defaults()
try:
    for key, value in DEFAULTS.items():
        dbutils.widgets.text(key, value)
    get_config = dbutils.widgets.get
except NameError:
    def get_config(key):
        return os.environ.get(key.upper(), DEFAULTS[key])

## One candidate, two storage precisions

At construction time, sum the piecewise-constant densities of the source histograms on their canonical boundary union.
Within each merged cell, compute the area-weighted density mean and variance, treating uncovered regions as zero.
Store `log(1 + variance / max(mean², 1e-18))`, or zero for a zero mean. This measures **source-estimated spatial
heterogeneity** using regularized log-CV², not raw-point uncertainty. It cannot recover point positions already missing from the sources.

The same scalar map is stored as float16 and float32. Queries decode only touched partial cells and derive two features:
the weighted mean (weights `(count+1)*overlap*(1-overlap)`) and maximum map value. Full-cell coverage contributes no
partial-cell metadata features. These are risk cues, not certified error bounds, and they lose within-cell arrangement.

Each binary histogram has a 24-byte header, float64 boundaries and float64 counts. Each map has a 24-byte header plus
2 or 4 bytes per cell. Actual file sizes define each budget. Maps remain encoded in runtime contexts. Model weights,
derived arrays and Python process memory are separate costs; this experiment does not claim a fixed process-memory limit.

In [ ]:
"""Query-independent source-density heterogeneity; not a true-error bound.

The field sums source histogram densities, treating each source as zero outside
its extent. One scalar per target cell loses within-cell spatial arrangement and
cannot recover unknown point positions inside the original source cells.

Binary artifact sizes are actual file sizes. Histogram payloads use float64;
metadata stays encoded as float16/float32 and only touched partial cells decode.
"""
from pathlib import Path
import struct

import numpy as np

_HIST_MAGIC = b'SLHIST01'
_META_MAGIC = b'SLMETA01'
_HIST_HEADER = struct.Struct('<IId')
_META_HEADER = struct.Struct('<IIB7x')
_E = 1e-9


def validate_hist(hist):
    if len(hist) != 4:
        raise ValueError('Histogram must be (rate, x boundaries, y boundaries, counts)')
    rate, bx, by, counts = hist
    bx, by, counts = np.asarray(bx, dtype=float), np.asarray(by, dtype=float), np.asarray(counts, dtype=float)
    if not np.isfinite(rate) or float(rate) < 0:
        raise ValueError('Invalid histogram rate')
    for boundaries in (bx, by):
        if (boundaries.ndim != 1 or len(boundaries) < 2 or not np.isfinite(boundaries).all()
                or not (np.diff(boundaries) > 0).all()):
            raise ValueError('Histogram boundaries must be finite and strictly increasing')
    if counts.shape != (len(bx)-1, len(by)-1) or not np.isfinite(counts).all() or (counts < 0).any():
        raise ValueError('Histogram counts must match the grid and be finite/nonnegative')
    return float(rate), bx, by, counts


def build_source_field(source_histograms):
    """Return exact piecewise-constant sum of the supplied source densities."""
    sources = [validate_hist(hist) for hist in source_histograms]
    if not sources:
        raise ValueError('At least one source histogram is required')
    ux = np.unique(np.concatenate([hist[1] for hist in sources]))
    uy = np.unique(np.concatenate([hist[2] for hist in sources]))
    field = np.zeros((len(ux)-1, len(uy)-1), dtype=float)
    for _, bx, by, counts in sources:
        density = counts / np.outer(np.diff(bx), np.diff(by))
        ix = np.flatnonzero((ux[:-1] >= bx[0]) & (ux[1:] <= bx[-1]))
        iy = np.flatnonzero((uy[:-1] >= by[0]) & (uy[1:] <= by[-1]))
        sx = np.searchsorted(bx, ux[ix], side='right') - 1
        sy = np.searchsorted(by, uy[iy], side='right') - 1
        field[np.ix_(ix, iy)] += density[np.ix_(sx, sy)]
    if not np.isfinite(field).all():
        raise ValueError('Source density overflow')
    return ux, uy, field


def _overlap_lengths(target, source):
    return np.maximum(0., np.minimum(target[1:, None], source[None, 1:])
                      - np.maximum(target[:-1, None], source[None, :-1]))


def metadata_values(field, target_histogram):
    """Area-weighted log(1+CV²) of source-estimated density per target cell.

    The full target area is the denominator, so uncovered area contributes zero.
    Numerical near-zero variances are clamped to zero and reported. No query or
    exact-answer information is used.
    """
    _, bx, by, counts = validate_hist(target_histogram)
    ux, uy, density = field
    _, ux, uy, density = validate_hist((1., ux, uy, density))
    wx, wy = _overlap_lengths(bx, ux), _overlap_lengths(by, uy)
    area = np.outer(np.diff(bx), np.diff(by))
    mass = wx @ density @ wy.T
    second = wx @ np.square(density) @ wy.T / area
    mean = mass / area
    variance = second - np.square(mean)
    if not (np.isfinite(mean).all() and np.isfinite(second).all()):
        raise ValueError('Density-moment overflow')
    tolerance = 64 * np.finfo(float).eps * np.maximum(second, np.square(mean))
    negative = variance < 0
    numerical = np.abs(variance) <= tolerance
    significant_negative = negative & ~numerical
    diagnostics = dict(negative_variance_cells=int(negative.sum()),
                       numerical_variance_clamped_cells=int(numerical.sum()),
                       significant_negative_variance_cells=int(significant_negative.sum()),
                       min_unclamped_variance=float(variance.min()),
                       source_estimated_mass=float(mass.sum()),
                       target_histogram_mass=float(counts.sum()),
                       source_vs_target_mass_gap=float(abs(mass.sum()-counts.sum())))
    # Integral arithmetic cannot create truly negative variance. Large violations
    # indicate an invalid numerical regime rather than evidence of homogeneity.
    if significant_negative.any():
        raise ValueError('Numerically invalid density variance: ' + str(diagnostics))
    variance = np.where(numerical, 0., np.maximum(variance, 0.))
    cv2 = variance / np.maximum(np.square(mean), _E**2)
    values = np.where(mean > 0, np.log1p(cv2), 0.)
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError('Invalid metadata values')
    diagnostics.update(max_log_cv2=float(values.max()), zero_metadata_cells=int((values == 0).sum()))
    return values, diagnostics


def save_hist(path, hist):
    rate, bx, by, counts = validate_hist(hist)
    path = Path(path)
    with path.open('wb') as stream:
        stream.write(_HIST_MAGIC)
        stream.write(_HIST_HEADER.pack(counts.shape[0], counts.shape[1], rate))
        for array in (bx, by, counts):
            stream.write(np.asarray(array, dtype='<f8').tobytes(order='C'))
    return path.stat().st_size


def load_hist_payload(path):
    payload = Path(path).read_bytes()
    header_size = len(_HIST_MAGIC) + _HIST_HEADER.size
    if len(payload) < header_size or payload[:8] != _HIST_MAGIC:
        raise ValueError('Invalid histogram artifact header')
    nx, ny, rate = _HIST_HEADER.unpack_from(payload, 8)
    expected = header_size + 8*((nx+1)+(ny+1)+nx*ny)
    if nx == 0 or ny == 0 or len(payload) != expected:
        raise ValueError('Invalid histogram artifact length')
    numbers = np.frombuffer(payload, dtype='<f8', offset=header_size)
    bx = numbers[:nx+1]
    by = numbers[nx+1:nx+ny+2]
    counts = numbers[nx+ny+2:].reshape(nx, ny)
    return validate_hist((rate, bx, by, counts))


def save_metadata(path, values, bits):
    values = np.asarray(values, dtype=float)
    if bits not in (16, 32) or values.ndim != 2 or min(values.shape, default=0) == 0:
        raise ValueError('Metadata needs a nonempty2D grid and bits16 or32')
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError('Metadata must be finite and nonnegative')
    encoded = values.astype('<f2' if bits == 16 else '<f4')
    if not np.isfinite(encoded).all():
        raise ValueError('Metadata encoding overflow')
    path = Path(path)
    with path.open('wb') as stream:
        stream.write(_META_MAGIC)
        stream.write(_META_HEADER.pack(values.shape[0], values.shape[1], bits))
        stream.write(encoded.tobytes(order='C'))
    return path.stat().st_size


def load_metadata(path):
    payload = Path(path).read_bytes()
    header_size = len(_META_MAGIC) + _META_HEADER.size
    if len(payload) < header_size or payload[:8] != _META_MAGIC:
        raise ValueError('Invalid metadata artifact header')
    nx, ny, bits = _META_HEADER.unpack_from(payload, 8)
    if bits not in (16, 32) or nx == 0 or ny == 0 or len(payload) != header_size+nx*ny*(bits//8):
        raise ValueError('Invalid metadata artifact length or encoding')
    encoded = np.frombuffer(payload, dtype='<f2' if bits == 16 else '<f4', offset=header_size).reshape(nx, ny)
    if not np.isfinite(encoded).all() or (encoded < 0).any():
        raise ValueError('Invalid encoded metadata')
    return encoded


def _fractions(boundaries, low, high):
    n = len(boundaries)-1
    first = int(np.clip(np.searchsorted(boundaries, low, side='right')-1, 0, n-1))
    last = int(np.clip(np.searchsorted(boundaries, high, side='left')-1, 0, n-1))
    if last < first:
        return first, last, np.empty(0)
    left, right = boundaries[first:last+1], boundaries[first+1:last+2]
    fraction = np.clip((np.minimum(high, right)-np.maximum(low, left))/(right-left), 0., 1.)
    return first, last, fraction


def metadata_query(q, bx, by, counts, encoded):
    """Query two risk features without accessing source histograms or raw data."""
    q = np.asarray(q, dtype=float)
    if q.shape != (4,) or not np.isfinite(q).all() or q[0] >= q[1] or q[2] >= q[3]:
        raise ValueError('Query must contain finite increasing rectangle bounds')
    bx, by, counts = np.asarray(bx), np.asarray(by), np.asarray(counts)
    encoded = np.asarray(encoded)
    if counts.shape != encoded.shape or counts.shape != (len(bx)-1, len(by)-1):
        raise ValueError('Metadata grid differs from histogram grid')
    if encoded.dtype.kind != 'f' or encoded.dtype.itemsize not in (2, 4):
        raise ValueError('Metadata must stay encoded as float16/float32')
    i0, i1, fx = _fractions(bx, q[0], q[1])
    j0, j1, fy = _fractions(by, q[2], q[3])
    zero = dict(meta_partial_mean=0., meta_partial_max=0.)
    if not fx.size or not fy.size:
        return zero
    overlap = np.outer(fx, fy)
    partial = (overlap > 0) & (overlap < 1-1e-12)
    if not partial.any():
        return zero
    # Decode touched partial cells only; never keep a full float64 sidecar copy.
    values = encoded[i0:i1+1, j0:j1+1][partial].astype(float)
    weights = (counts[i0:i1+1, j0:j1+1][partial]+1) * overlap[partial] * (1-overlap[partial])
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError('Invalid touched metadata values')
    return dict(meta_partial_mean=float(np.dot(weights, values)/weights.sum()),
                meta_partial_max=float(values.max()))

## Comparisons and safeguards

| Variant | Information available to its runtime feature extractor |
|---|---|
| `merged_only` |14 query/merged inputs, excluding SA and all source overlap/resolution features |
| `full_source` |The same 14 inputs plus 7 source features; newly trained 21-input reference, not saved B24 |
| `compact16`, `compact32` |The same 14 inputs plus 2 partial-cell metadata features; no source contexts |
| `finer16`, `finer32` |14 inputs on the finest **available ARM export** fitting the corresponding compact total budget |

Logged SA and merged-count-width scores are calibrated quality references on the base histogram. SA scores are read
from verified logger outputs, so this notebook does not report a query-time SA latency. All learned variants use the
existing LightGBM regression settings and training-only early stopping. Every representation gets its own calibration
gate, selected without test outcomes. The error limit is 25%, the empirical accepted-violation target 5%; this is not
a population guarantee. Zero accepted answers means undefined risk.

Finer exports are chosen by bytes only. A fine histogram has its own estimate, error label and gate. Repeated mappings
to the same physical histogram are deduplicated for training. Identical training datasets may reuse a fit, with provenance.
`budgets.csv` records unused space and same-H fallbacks. This discrete-export baseline is not the optimal possible
exact-budget finer histogram; a convincing result may require denser resolution baselines later.

The query split exactly reuses 08d. Optional cell split groups queries by the existing source-cell mapping and is reported
separately; it is not cross-dataset transfer or strict non-overlapping geographic holdout. Existing test reuse is exploratory.
The cell IDs are offline evaluation metadata, never model inputs. No budgets or feature definitions are chosen using test quality.
The existing 08d cohort requires an exact count of at least 10. Relative error is `abs(estimate - truth) / truth`;
small-count and zero-count queries are excluded, so this experiment makes no quality claim for them.

## Provenance and input resolution

In [ ]:
def sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False, default=str))
    temporary.replace(path)


def choose_ids(ids, limit, salt):
    ordered = sorted(map(int, set(ids)), key=lambda q: hashlib.sha256(f'{salt}:{q}'.encode()).digest())
    return sorted(ordered[:limit])


def load_settings():
    cfg = {key: str(get_config(key)).strip() for key in DEFAULTS}
    for key in ('max_trees', 'num_threads', 'timing_queries', 'timing_repeats', 'random_seed'):
        cfg[key] = int(cfg[key])
    if min(cfg['max_trees'], cfg['num_threads'], cfg['timing_queries'], cfg['timing_repeats']) < 1:
        raise ValueError('Tree/thread/timing settings must be positive')
    cfg['seeds'] = [int(x) for x in cfg['seeds'].split(',')]
    cfg['splits'] = cfg['splits'].split(',')
    if (not set(cfg['splits']) <= {'query', 'cell'} or len(set(cfg['splits'])) != len(cfg['splits'])
            or len(set(cfg['seeds'])) != len(cfg['seeds'])):
        raise ValueError('Use unique seeds and query/cell splits')
    return cfg


def resolve_source(cfg):
    if cfg['run_dir']:
        source = Path(cfg['run_dir']).expanduser().resolve()
    else:
        candidates = []
        for path in Path(cfg['output_root']).expanduser().glob('*/run.json'):
            info = json.loads(path.read_text())
            if info.get('status') == 'complete' and info.get('study') == 'source-information':
                candidates.append(path)
        if not candidates:
            raise FileNotFoundError('Set run_dir to a completed 08d directory with run.json, splits.csv and matching histogram exports.')
        source = max(candidates, key=lambda p: p.stat().st_mtime_ns).parent
    prior = json.loads((source / 'run.json').read_text())
    if prior.get('status') != 'complete' or prior.get('study') != 'source-information':
        raise ValueError('run_dir must be a completed 08d source-information run, not 09 or a ZIP')
    inputs = {}
    for key, expected in prior['input_sha256'].items():
        group, filename = key.split('/', 1)
        override = cfg['data_dir'] if group == 'data' else cfg['histogram_dir']
        options = [Path(override) / filename] if override else [source / 'inputs' / key]
        if not override:
            original = prior['config']['data_root' if group == 'data' else 'sample_root']
            options.append(Path(original) / prior['config']['dataset'] / filename)
        path = next((p for p in options if p.is_file()), None)
        if path is None:
            raise FileNotFoundError('Missing input ' + key + '; set data_dir/histogram_dir to matching inputs')
        if sha(path) != expected:
            raise ValueError('Input changed since 08d: ' + str(path))
        inputs[key] = path.resolve()
    print('Using 08d source:', source, flush=True)
    return source, prior, inputs


def import_existing_helpers(cfg, prior):
    repo = Path(cfg['repo_root']).resolve()
    import sys
    sys.path.insert(0, str(repo / 'ml'))
    import features as merged_helpers
    import features_v1 as source_helpers
    import gate_eval as gate_helpers
    import model_study as study_helpers
    for module in (merged_helpers, source_helpers, gate_helpers, study_helpers):
        if Path(module.__file__).resolve().parent != repo / 'ml':
            raise ValueError('Another checkout is imported; restart Python')
        importlib.reload(module)
    for name in ('features.py', 'features_v1.py'):
        if sha(repo / 'ml' / name) != prior['code_sha256'][name]:
            raise ValueError('Feature definitions differ from 08d: ' + name)
    return merged_helpers, source_helpers, gate_helpers, study_helpers

## Runtime feature providers: restricted to explicit inputs

In [ ]:
def histogram_context(hist):
    rate, bx, by, counts = validate_hist(hist)
    area = np.outer(np.diff(bx), np.diff(by))
    density = counts / area
    local = np.abs(np.log((density + 1e-9) / (uniform_filter(density, size=3, mode='nearest') + 1e-9)))
    nonzero = density[counts > 0]
    return dict(rate=rate, bx=bx, by=by, C=counts, area=area, LC=local,
                median=float(np.median(nonzero)) if len(nonzero) else 0., total=float(counts.sum()))


def base_query(q, context, helper):
    f = helper._query_features(q, context['bx'], context['by'], context['C'], context['area'],
                              context['LC'], context['median'], context['total'], [])
    f['merge_rate'] = context['rate']
    return f


def source_query(q, contexts, helper):
    # Same weighting/formulas as features_v1.source_features, applied to one live query.
    est = full = width = contrast = in_out = empty = partial = 0.
    for ctx in contexts:
        if not (ctx['bx'][0] < q[1] and q[0] < ctx['bx'][-1] and ctx['by'][0] < q[3] and q[2] < ctx['by'][-1]):
            continue
        f = base_query(q, ctx, helper)
        e = f['est_py']; est += e
        full += np.expm1(f['log_full'])
        width += f['bound_rel'] * max(e, 1.)
        contrast += e * f['edge_contrast']; in_out += e * f['in_out']
        empty += e * f['empty_partial']; partial += f['n_partial']
    weight = max(est, 1e-9)
    return dict(src_log_est=np.log1p(est), src_bound_rel=width/max(est, 1.),
                src_edge_share=(est-full)/weight, src_edge_contrast=contrast/weight,
                src_in_out=in_out/weight, src_empty_partial=empty/weight, src_n_partial=partial)


def extract_query(q, context, helper, metadata=None, sources=None):
    if metadata is not None and sources is not None:
        raise ValueError('Compact provider cannot receive full source contexts')
    f = base_query(q, context, helper)
    if metadata is not None:
        f.update(metadata_query(q, context['bx'], context['by'], context['C'], metadata))
    if sources is not None:
        f.update(source_query(q, sources, helper))
    return f


def frame_for_context(qids, queries, context, helper):
    rows = []
    for qid in qids:
        q = queries[qid, :4]
        f = base_query(q, context, helper)
        estimate, truth = float(f.pop('est_py')), float(queries[qid, 4])
        error = abs(estimate-truth)/truth
        rows.append(dict(qId=qid, physical_rate=context['rate'], estimate=estimate, truth=truth,
                         relErr=error, label=math.log(max(error, .001)), **f))
    return pd.DataFrame(rows)

## Build artifacts using source histograms only; labels do not enter metadata construction

In [ ]:
def prepare_artifacts(cfg, destination, state):
    source, prior, inputs = resolve_source(cfg)
    merged_helpers, source_helpers, gate_helpers, study_helpers = import_existing_helpers(cfg, prior)
    stage = Path(tempfile.mkdtemp(prefix='synopsislake-compact-'))
    for group in ('data', 'run'):
        (stage / group).mkdir()
    for key, path in inputs.items():
        if key != 'run/rows.csv':
            shutil.copyfile(path, stage / key)
            if sha(stage / key) != prior['input_sha256'][key]:
                raise ValueError('Input changed during staging')
    queries = pd.read_csv(stage / 'data/queries-Ans.txt', header=None).to_numpy(float)
    if queries.shape[1] != 5 or not np.isfinite(queries).all():
        raise ValueError('Invalid query input')
    if ((queries[:, 0] >= queries[:, 1]) | (queries[:, 2] >= queries[:, 3])).any():
        raise ValueError('Invalid query rectangle')
    qids = np.flatnonzero(queries[:, 4] >= 10).tolist()
    splits = pd.read_csv(source / 'splits.csv')
    if splits.duplicated(['seed', 'qId']).any():
        raise ValueError('Duplicate split identifiers')
    for seed in cfg['seeds']:
        old = splits[splits.seed == seed]
        if set(old.qId) != set(qids):
            raise ValueError('Saved 08d eligible cohort differs; do not resplit silently')
        for label, ids in zip(('train', 'calibration', 'test'), gate_helpers.split_ids(qids, seed)):
            if ids != set(old.loc[old.split == label, 'qId']):
                raise ValueError('Saved query split differs from current helper')
    payload_dir = destination / 'payloads'; payload_dir.mkdir()
    original_hists = {p.stem: merged_helpers.load_hist(p) for p in sorted((stage / 'run').glob('arm_*.txt'))}
    if not original_hists:
        raise ValueError('Missing ARM exports')
    paths, contexts = {}, {}
    for name, hist in original_hists.items():
        path = payload_dir / (name + '.hist')
        save_hist(path, hist); paths[name] = path
    rates = {name: float(hist[0]) for name, hist in original_hists.items()}
    if cfg['merge_rates'].lower() == 'all':
        base_names = sorted(rates, key=rates.get)
    else:
        requested = [float(x) for x in cfg['merge_rates'].split(',')]
        base_names = sorted({min(rates, key=lambda n: abs(rates[n]-r)) for r in requested}, key=rates.get)
    cfg['resolved_base_rates'] = [rates[n] for n in base_names]
    source_paths = []
    source_hists = [merged_helpers.load_hist(p) for p in sorted((stage / 'run').glob('src_*.txt'))]
    for i, hist in enumerate(source_hists):
        path = payload_dir / f'src_{i}.hist'; save_hist(path, hist); source_paths.append(path)
    tick = time.perf_counter(); field = build_source_field(source_hists)
    state['timings']['source_field_build_seconds'] = time.perf_counter()-tick
    metadata, mapping, budgets, diagnostics = {}, {}, [], []
    for name in base_names:
        tick = time.perf_counter(); values, detail = metadata_values(field, original_hists[name])
        detail.update(histogram=name, mergeRate=rates[name], build_seconds=time.perf_counter()-tick)
        diagnostics.append(detail)
        for bits in (16, 32):
            path = payload_dir / f'{name}-cv{bits}.meta'; save_metadata(path, values, bits)
            metadata[(name, bits)] = path
            budget = paths[name].stat().st_size + path.stat().st_size
            feasible = [n for n in paths if paths[n].stat().st_size <= budget and rates[n] >= rates[name]-1e-12]
            # Resolution choice depends on artifact bytes only, never queries, scores or truth.
            fine = max(feasible, key=lambda n: (original_hists[n][3].size, rates[n]))
            mapping[(name, bits)] = fine
            budgets.append(dict(base_histogram=name, base_rate=rates[name], bits=bits,
                base_bytes=paths[name].stat().st_size, metadata_bytes=path.stat().st_size, total_budget_bytes=budget,
                finer_histogram=fine, finer_rate=rates[fine], finer_bytes=paths[fine].stat().st_size,
                unused_finer_budget_bytes=budget-paths[fine].stat().st_size, strictly_finer=(fine != name),
                finer_candidate_count=len(feasible), finer_baseline='largest available ARM export within budget'))
    pd.DataFrame(budgets).to_csv(destination / 'budgets.csv', index=False)
    pd.DataFrame(diagnostics).to_csv(destination / 'metadata_build.csv', index=False)
    needed = set(base_names) | set(mapping.values())
    load_rows=[]
    for name in sorted(needed):
        tick=time.perf_counter()
        contexts[name] = histogram_context(load_hist_payload(paths[name]))
        load_rows.append(dict(artifact=name,kind='histogram_context',load_seconds=time.perf_counter()-tick))
    encoded_maps={}
    for key,path in metadata.items():
        tick=time.perf_counter();encoded_maps[key]=load_metadata(path)
        load_rows.append(dict(artifact=path.name,kind='encoded_metadata',load_seconds=time.perf_counter()-tick))
    # Build reference features once per query offline; query-time costs are measured separately.
    tick = time.perf_counter()
    full_source = source_helpers.source_features(str(stage / 'run'), str(stage / 'data')).set_index('qId')
    state['timings']['offline_source_features_seconds'] = time.perf_counter()-tick
    tick=time.perf_counter()
    source_contexts = [histogram_context(load_hist_payload(path)) for path in source_paths]
    load_rows.append(dict(artifact='all_sources',kind='source_contexts',load_seconds=time.perf_counter()-tick))
    pd.DataFrame(load_rows).to_csv(destination/'payload_loads.csv',index=False)
    for qid in choose_ids(qids, min(20, len(qids)), cfg['random_seed']):
        live = source_query(queries[qid, :4], source_contexts, merged_helpers)
        for name, value in live.items():
            if not np.isclose(value, full_source.loc[qid, name], rtol=1e-10, atol=1e-10):
                raise ValueError('Live source feature differs from existing implementation: ' + name)
    tables = {}
    tick = time.perf_counter()
    for name in sorted(needed):
        tables[name] = frame_for_context(qids, queries, contexts[name], merged_helpers)
    state['timings']['offline_base_features_seconds'] = time.perf_counter()-tick
    logged = pd.read_csv(inputs['run/rows.csv'], usecols=['method', 'mergeRate', 'qId', 'est', 'predErr'])
    if sha(inputs['run/rows.csv']) != prior['input_sha256']['run/rows.csv']:
        raise ValueError('Logger input changed during verification')
    logger_gaps = []
    for name, table in tables.items():
        old = logged[(logged.method == 'ARM') & np.isclose(logged.mergeRate, rates[name], rtol=0, atol=1e-12)]
        joined = table[['qId', 'estimate']].merge(old[['qId', 'est', 'predErr']], on='qId', how='left', validate='one_to_one')
        if joined.est.isna().any() or float((joined.estimate-joined.est).abs().max()) > .001:
            raise ValueError('Histogram estimator differs from existing Java export: ' + name)
        table['logged_sa'] = joined.predErr.to_numpy()
        logger_gaps.append(dict(histogram=name, max_estimate_gap=float((joined.estimate-joined.est).abs().max())))
    pd.DataFrame(logger_gaps).to_csv(destination / 'estimate_parity.csv', index=False)
    # Group identifiers are offline evaluation metadata, never predictor inputs.
    cells = study_helpers.source_cells(str(stage / 'run'), str(stage / 'data')).set_index('qId').cell
    base_features = [x for x in merged_helpers.FEATURES if x not in
        {'skew_pred', 'log_alpha', 'n_src_overlap', 'src_res_min', 'src_res_max'}]
    if len(base_features) != 14:
        raise ValueError('Unexpected merged-only feature definition')
    definitions = {'merged_only': base_features, 'full_source': base_features + list(source_helpers.SOURCE_FEATURES),
        'compact16': base_features + ['meta_partial_mean', 'meta_partial_max'],
        'compact32': base_features + ['meta_partial_mean', 'meta_partial_max'],
        'finer16': base_features, 'finer32': base_features}
    frames, routes, payload_rows = {}, {}, []
    meta_tables = {}
    for variant in definitions:
        pieces, route = [], {}
        for base in base_names:
            bits = int(variant[-2:]) if variant.startswith(('compact', 'finer')) else None
            physical = mapping[(base, bits)] if variant.startswith('finer') else base
            route[base] = physical
            table = tables[physical].copy()
            table['histogram_key'] = physical
            if variant == 'full_source':
                table = table.join(full_source[source_helpers.SOURCE_FEATURES], on='qId', validate='many_to_one')
            if variant.startswith('compact'):
                key = (base, bits)
                if key not in meta_tables:
                    encoded = encoded_maps[key]
                    meta_tables[key] = pd.DataFrame([metadata_query(queries[qid, :4], contexts[base]['bx'],
                        contexts[base]['by'], contexts[base]['C'], encoded) for qid in qids], index=qids)
                table = table.join(meta_tables[key], on='qId', validate='many_to_one')
            pieces.append(table)
            hist_bytes = paths[physical].stat().st_size
            meta_bytes = metadata[(base, bits)].stat().st_size if variant.startswith('compact') else 0
            src_bytes = sum(p.stat().st_size for p in source_paths) if variant == 'full_source' else 0
            ctx = contexts[physical]
            derived = sum(ctx[k].nbytes for k in ('bx', 'by', 'C', 'area', 'LC'))
            payload_rows.append(dict(variant=variant, base_histogram=base, base_rate=rates[base], physical_histogram=physical,
                physical_rate=rates[physical], histogram_bytes=hist_bytes, metadata_bytes=meta_bytes,
                source_bytes=src_bytes, total_synopsis_payload_bytes=hist_bytes+meta_bytes+src_bytes,
                source_histograms_required=len(source_paths) if variant == 'full_source' else 0,
                metadata_bits=bits if variant.startswith('compact') else 0,
                derived_numpy_context_bytes=derived + (sum(sum(c[k].nbytes for k in ('bx','by','C','area','LC')) for c in source_contexts) if variant == 'full_source' else 0),
                encoded_metadata_array_bytes=encoded_maps[(base,bits)].nbytes if variant.startswith('compact') else 0,
                model_weights_included=False, python_objects_and_temporaries_included=False))
        # Finer mappings may alias physical histograms: train each query/physical-H pair once.
        frames[variant] = pd.concat(pieces, ignore_index=True).drop_duplicates(['qId', 'histogram_key']).reset_index(drop=True)
        if not np.isfinite(frames[variant][definitions[variant] + ['label', 'relErr']].to_numpy(float)).all():
            raise ValueError('Nonfinite features or labels for ' + variant)
        routes[variant] = route
    pd.DataFrame(payload_rows).to_csv(destination / 'payloads.csv', index=False)
    save_json(destination / 'feature_definitions.json', definitions)
    save_json(destination / 'input_manifest.json', {key: dict(path=str(p), sha256=sha(p)) for key, p in inputs.items()})
    state.update(source_run=str(source), source_manifest_sha256=sha(source / 'run.json'),
        source_code_sha256={n: sha(Path(cfg['repo_root'])/'ml'/n) for n in ('features.py','features_v1.py','gate_eval.py','model_study.py')},
        package_versions={name: importlib.metadata.version(name) for name in ('lightgbm','numpy','pandas','scipy','scikit-learn')},
        eligible_queries=len(qids), excluded_small_truth=len(queries)-len(qids), variants=list(definitions),
        feature_counts={v:len(f) for v,f in definitions.items()}, staging_directory=str(stage))
    return dict(frames=frames, routes=routes, contexts=contexts, metadata=metadata, encoded_maps=encoded_maps, paths=paths,
                source_paths=source_paths, source_contexts=source_contexts, cells=cells, splits=splits,
                queries=queries, base_names=base_names, rates=rates, definitions=definitions,
                merged_helpers=merged_helpers, gate_helpers=gate_helpers)

## Train and calibrate without test-guided decisions

In [ ]:
def evaluate_score(cal, test, cal_score, test_score, epsilon=.25, delta=.05):
    tick=time.perf_counter()
    threshold = float(HELPERS.cutoff(np.asarray(cal_score), cal.relErr.to_numpy(), epsilon, delta))
    calibration_gate_seconds=time.perf_counter()-tick
    cal_accepted=np.asarray(cal_score)<=threshold
    cal_bad=int((cal_accepted & (cal.relErr.to_numpy()>epsilon)).sum())
    accepted = np.asarray(test_score) <= threshold
    bad = test.relErr.to_numpy() > epsilon
    n, violations = int(accepted.sum()), int((accepted & bad).sum())
    good = int((~bad).sum())
    return dict(cutoff=threshold if math.isfinite(threshold) else np.nan, accept_none=not math.isfinite(threshold),
        calibration_accepted_count=int(cal_accepted.sum()),calibration_violation_count=cal_bad,
        calibration_violation=cal_bad/int(cal_accepted.sum()) if cal_accepted.any() else np.nan,
        calibration_gate_seconds=calibration_gate_seconds,
        test_count=len(test), accepted_count=n, violation_count=violations, good_accepted_count=n-violations,
        true_good_count=good, acceptance=n/len(test), violation=violations/n if n else np.nan,
        good_recall=(n-violations)/good if good else np.nan,
        quality_target_met=(violations/n <= delta) if n else None,
        auc=roc_auc_score(bad, test_score) if bad.any() and (~bad).any() else np.nan)


def fixed_coverage(test, scores, fractions=(.05, .10)):
    rows = []
    for fraction in fractions:
        count = max(1, int(math.ceil(fraction*len(test))))
        threshold = float(np.partition(np.asarray(scores), count-1)[count-1])
        accepted = np.asarray(scores) <= threshold
        bad = test.relErr.to_numpy() > .25
        n = int(accepted.sum())
        rows.append(dict(nominal_coverage=fraction, actual_coverage=n/len(test), nominal_count=count,
            score_cutoff=threshold,accepted_count=n, tie_extra_count=n-count, violation_count=int((accepted & bad).sum()),
            violation=float(bad[accepted].mean()), test_count=len(test), diagnostic_only=True,
            cutoff_source='test scores only; truth used only to score the selected answers'))
    return rows


def run_models(cfg, destination, state, data):
    global HELPERS
    HELPERS = data['gate_helpers']
    quality, fixed, models, split_rows, predictions = [], [], [], [], []
    cache, live_models = {}, {}
    qids = sorted(set(data['frames']['merged_only'].qId))
    for split_kind in cfg['splits']:
        for seed in cfg['seeds']:
            if split_kind == 'query':
                old = data['splits'][data['splits'].seed == seed]
                groups = tuple(set(old.loc[old.split == name, 'qId']) for name in ('train','calibration','test'))
            else:
                train, cal, test = HELPERS.split_ids(data['cells'].loc[qids].unique(), seed)
                groups = tuple(set(data['cells'].loc[qids][data['cells'].loc[qids].isin(ids)].index)
                               for ids in (train, cal, test))
            if any(len(group) < 3 for group in groups) or set.union(*groups) != set(qids):
                raise ValueError('Insufficient or incomplete split; do not repair it using test outcomes')
            if any(groups[i] & groups[j] for i in range(3) for j in range(i)):
                raise ValueError('Overlapping query splits')
            for label, ids in zip(('train','calibration','test'), groups):
                split_rows.extend(dict(split_kind=split_kind, seed=seed, qId=q, split=label,
                                       cell=data['cells'].loc[q]) for q in sorted(ids))
            for variant, names in data['definitions'].items():
                frame = data['frames'][variant]
                train = frame[frame.qId.isin(groups[0])]
                folder = destination / 'models' / split_kind / f'seed_{seed}' / variant
                folder.mkdir(parents=True)
                identity=train[['qId','histogram_key',*names,'label']].sort_values(['histogram_key','qId'])
                training_digest=hashlib.sha256(pd.util.hash_pandas_object(identity,index=False).to_numpy().tobytes()).hexdigest()
                signature = (split_kind,seed,tuple(names),training_digest)
                tick = time.perf_counter()
                if signature in cache:
                    model, original = cache[signature]; fit_seconds = 0.; reused = original
                else:
                    params = dict(HELPERS.PARAMS, n_estimators=cfg['max_trees'], n_jobs=cfg['num_threads'])
                    model = HELPERS.fit(train, names, seed, params=params)
                    fit_seconds = time.perf_counter()-tick; reused = None
                    cache[signature] = (model, variant)
                model.booster_.save_model(str(folder/'model.txt'))
                live_models[(split_kind, seed, variant)] = model.booster_
                save_json(folder/'schema.json', dict(features=names, split_kind=split_kind, seed=seed, variant=variant,
                    label='log(max(relative_error,0.001))', min_exact=10, params=dict(HELPERS.PARAMS,n_estimators=cfg['max_trees'],n_jobs=cfg['num_threads'])))
                models.append(dict(split_kind=split_kind,seed=seed,variant=variant,fit_seconds=fit_seconds,
                    reused_from=reused,tree_count=model.booster_.num_trees(),model_bytes=(folder/'model.txt').stat().st_size,
                    training_query_histogram_rows=len(train),model_sha256=sha(folder/'model.txt')))
                for base in data['base_names']:
                    physical = data['routes'][variant][base]
                    part = frame[frame.histogram_key == physical]
                    cal, test = (part[part.qId.isin(ids)] for ids in groups[1:])
                    tick = time.perf_counter()
                    cs, ts = model.predict(cal[names],num_threads=cfg['num_threads']), model.predict(test[names],num_threads=cfg['num_threads'])
                    if not np.isfinite(cs).all() or not np.isfinite(ts).all():
                        raise ValueError('Nonfinite model scores')
                    prefix = dict(split_kind=split_kind,seed=seed,variant=variant,base_rate=data['rates'][base],physical_rate=data['rates'][physical])
                    evaluation = evaluate_score(cal,test,cs,ts)
                    quality.append(dict(**prefix, **evaluation, score='learned',
                                        score_space='predicted log-relative-error',calibration_count=len(cal), batch_prediction_seconds=time.perf_counter()-tick))
                    fixed.extend(dict(**prefix, **row, score='learned') for row in fixed_coverage(test,ts))
                    threshold = -np.inf if evaluation['accept_none'] else evaluation['cutoff']
                    prediction = test[['qId','estimate','truth','relErr']].assign(score_log=ts, accepted=ts<=threshold,
                        **prefix)
                    predictions.append(prediction)
                    if variant == 'merged_only':
                        for score, column in [('logged_sa','logged_sa'),('merged_bound','bound_rel')]:
                            metric = evaluate_score(cal,test,cal[column].to_numpy(),test[column].to_numpy())
                            quality.append(dict(**prefix, **metric, score=score,score_space='logged SA' if score=='logged_sa' else 'relative count width',calibration_count=len(cal), batch_prediction_seconds=np.nan))
                            fixed.extend(dict(**prefix, **row, score=score) for row in fixed_coverage(test,test[column].to_numpy()))
                print(split_kind, 'seed',seed,variant, 'fit seconds:',round(fit_seconds,2),flush=True)
    pd.DataFrame(quality).to_csv(destination/'quality_by_setting.csv',index=False)
    pd.DataFrame(fixed).to_csv(destination/'fixed_coverage.csv',index=False)
    pd.DataFrame(models).to_csv(destination/'models.csv',index=False)
    pd.DataFrame(split_rows).to_csv(destination/'splits.csv',index=False)
    prediction_table = pd.concat(predictions,ignore_index=True)
    prediction_table.to_csv(destination/'test_predictions.csv',index=False)
    # Keep the actual batch results and calibration gates for checks outside the microtimer.
    keys = ['split_kind','seed','variant','base_rate']
    data['frozen_predictions'] = prediction_table.set_index(keys+['qId'])
    data['frozen_gates'] = pd.DataFrame(quality).query("score == 'learned'").set_index(keys)
    return pd.DataFrame(quality), pd.DataFrame(fixed), live_models, pd.DataFrame(split_rows)

## Single-query resident cost checks: not an end-to-end execution benchmark

In [ ]:
def time_providers(cfg, destination, data, live_models, split_table):
    rows = []
    indexed={v:f.set_index(['histogram_key','qId']) for v,f in data['frames'].items()}
    rng = np.random.default_rng(cfg['random_seed'])
    for split_kind in cfg['splits']:
        for seed in cfg['seeds']:
            ids = split_table[(split_table.split_kind == split_kind) & (split_table.seed == seed) & (split_table.split == 'test')].qId
            sample = choose_ids(ids, cfg['timing_queries'], cfg['random_seed']+seed)
            for base in data['base_names']:
                for repeat in range(cfg['timing_repeats']+1):
                    for qid in rng.permutation(sample):
                        q = data['queries'][qid,:4]
                        for variant in rng.permutation(list(data['definitions'])):
                            physical = data['routes'][variant][base]
                            context = data['contexts'][physical]
                            encoded = data['encoded_maps'][(base,int(variant[-2:]))] if variant.startswith('compact') else None
                            # Loading/deserialization is reusable setup here, excluded from resident microtimings.
                            sources = data['source_contexts'] if variant == 'full_source' else None
                            model = live_models[(split_kind,seed,variant)]
                            names = data['definitions'][variant]
                            tick = time.perf_counter_ns()
                            features = extract_query(q,context,data['merged_helpers'],metadata=encoded,sources=sources)
                            x = np.array([[features[name] for name in names]],dtype=float)
                            after_features = time.perf_counter_ns()
                            score = float(model.predict(x,num_threads=cfg['num_threads'])[0])
                            after_prediction = time.perf_counter_ns()
                            expected = indexed[variant].loc[(physical,qid),names].to_numpy(float)
                            if not np.allclose(x[0],expected,rtol=1e-9,atol=1e-9):
                                raise ValueError('Live provider differs from training features')
                            if not math.isfinite(score):
                                raise ValueError('Nonfinite live prediction')
                            key = (split_kind,seed,variant,data['rates'][base])
                            reference = data['frozen_predictions'].loc[(*key,qid)]
                            gate = data['frozen_gates'].loc[key]
                            cutoff = -np.inf if gate.accept_none else float(gate.cutoff)
                            accepted = score <= cutoff
                            # Tiny feature differences can change a tree path: check scores AND decisions.
                            if not np.isclose(score,reference.score_log,rtol=1e-8,atol=1e-10):
                                raise ValueError('Live prediction differs from saved batch prediction')
                            if accepted != bool(reference.accepted):
                                raise ValueError('Live decision differs from the frozen calibration gate')
                            if repeat:
                                rows.append(dict(split_kind=split_kind,seed=seed,base_rate=data['rates'][base],variant=variant,
                                    qId=qid,repeat=repeat-1,feature_ns=after_features-tick,inference_ns=after_prediction-after_features,
                                    total_ns=after_prediction-tick,source_contexts_available=len(sources) if sources is not None else 0,
                                    score_log=score,reference_score_log=float(reference.score_log),
                                    cutoff=float(gate.cutoff),accept_none=bool(gate.accept_none),
                                    accepted=bool(accepted),decision_match=True))
    pd.DataFrame(rows).to_csv(destination/'resident_timings.csv',index=False)
    group = pd.DataFrame(rows).groupby(['split_kind','seed','base_rate','variant'])
    summary = group.agg(observations=('qId','size'),query_count=('qId','nunique'),
        feature_mean_ms=('feature_ns',lambda x:x.mean()/1e6),feature_p95_ms=('feature_ns',lambda x:x.quantile(.95)/1e6),
        inference_mean_ms=('inference_ns',lambda x:x.mean()/1e6),inference_p95_ms=('inference_ns',lambda x:x.quantile(.95)/1e6),
        total_mean_ms=('total_ns',lambda x:x.mean()/1e6),total_p95_ms=('total_ns',lambda x:x.quantile(.95)/1e6)).reset_index()
    summary['scope']='resident Python feature+native inference only; no retrieval, routing or fallback'
    summary.to_csv(destination/'resident_timing_summary.csv',index=False)


def export_compact_results(destination):
    with ZipFile(destination/'compact_metadata_results.zip','w',ZIP_DEFLATED) as archive:
        for path in destination.rglob('*'):
            if path.is_file() and path.suffix in {'.csv','.json','.md','.png'} and 'models' not in path.relative_to(destination).parts:
                archive.write(path,path.relative_to(destination))


def run_prototype(cfg):
    destination = Path(cfg['output_root']).expanduser().resolve()/('compact-quality-'+uuid4().hex[:12])
    destination.mkdir(parents=True,exist_ok=False)
    state = dict(status='running',study='compact-quality',config=cfg,timings={},started_utc=datetime.now(timezone.utc).isoformat(),
        python=platform.python_version(),epsilon=.25,delta=.05,
        notes=['Regularized source-estimated density CV is not true within-cell error or an error bound.',
               'All learned models exclude SA and full-source geometry metadata from the 14 common inputs.',
               'Full-source 21-input reference is retrained, not the earlier B24 model.',
               'Existing inspected queries remain exploratory; no fresh holdout or universal risk claim.',
               'Empirical gates selected only on calibration; test truth is never used to select budgets or finer exports.',
               'Finer baseline is restricted to available exports; slack and same-H fallbacks are reported.',
               'Serialized synopsis bytes exclude model weights and runtime contexts, reported separately.',
               'This process holds all comparison variants; per-provider bytes are not process peak memory.',
               'SA scores are logged offline references, with no measured query-time SA latency.',
               'Resident microtimings do not establish retrieval savings or end-to-end speedup.'])
    save_json(destination/'run.json',state); started=time.perf_counter()
    try:
        tick=time.perf_counter();data=prepare_artifacts(cfg,destination,state)
        state['timings']['artifact_and_feature_preparation_seconds']=time.perf_counter()-tick
        save_json(destination/'run.json',state)
        tick=time.perf_counter();quality,fixed,models,splits=run_models(cfg,destination,state,data)
        state['timings']['training_calibration_evaluation_seconds']=time.perf_counter()-tick
        tick=time.perf_counter();time_providers(cfg,destination,data,models,splits)
        state['timings']['resident_cost_validation_seconds']=time.perf_counter()-tick
        learned=quality[quality.score=='learned']
        pairs=[]
        key=['split_kind','seed','base_rate']
        for variant,reference in [('compact16','merged_only'),('compact16','finer16'),('compact32','merged_only'),('compact32','finer32')]:
            left=learned[learned.variant==variant];right=learned[learned.variant==reference]
            joined=left.merge(right,on=key,suffixes=('_candidate','_reference'),validate='one_to_one')
            for row in joined.to_dict('records'):
                pairs.append(dict(**{k:row[k] for k in key},candidate=variant,reference=reference,
                    acceptance_delta=row['acceptance_candidate']-row['acceptance_reference'],
                    good_accepted_delta=row['good_accepted_count_candidate']-row['good_accepted_count_reference'],
                    violation_delta=row['violation_candidate']-row['violation_reference'],
                    candidate_violation=row['violation_candidate'],reference_violation=row['violation_reference']))
        pd.DataFrame(pairs).to_csv(destination/'paired_deltas.csv',index=False)
        summary=quality.groupby(['split_kind','variant','score'],as_index=False).agg(
            settings=('base_rate','size'),acceptance_mean=('acceptance','mean'),violation_mean_defined=('violation','mean'),
            accepted_cases=('accepted_count','sum'),violation_cases=('violation_count','sum'),good_accepted_cases=('good_accepted_count','sum'),
            auc_mean=('auc','mean'))
        summary['pooled_case_violation']=np.where(summary.accepted_cases>0,summary.violation_cases/summary.accepted_cases,np.nan)
        summary.to_csv(destination/'summary.csv',index=False)
        state.update(status='complete',wall_seconds=time.perf_counter()-started,finished_utc=datetime.now(timezone.utc).isoformat())
        save_json(destination/'run.json',state)
        (destination/'analysis_en.md').write_text(
            '# Compact metadata experiment\n\nCompare query/cell splits separately. Use quality_by_setting.csv for the primary '
            'per-seed/per-base-rate acceptance, counts, risk and AUC; zero acceptance has undefined risk. Fixed-coverage '
            'diagnostics use test scores only and retain complete ties; they are not deployment gates.\n\n'
            'Compare compact16/32 with merged_only, the full_source information reference and matching finer16/32 '
            'within actual serialized byte budgets. The finer candidate set contains existing ARM exports only; '
            'a same-H fallback or large unused budget weakens any equal-budget conclusion. Use budgets.csv.\n\n'
            'Metadata stores regularized log(1+CV²) of SOURCE-estimated summed density, not raw point positions. It may be zero '
            'at full source resolution even when query answers are inaccurate. Only partial cells are decoded at runtime. '
            'A scalar map loses spatial arrangement; usefulness is an experimental question.\n\n'
            'All methods share 14 source-free query/merged features. Compact models add 2 features; full_source adds 7 '
            'source features and is not a reused 08d B24 model. Finer models use their own estimates, labels and calibrated '
            'gates. Duplicate physical query/histogram training rows are removed, and identical feature datasets reuse '
            'fits with provenance.\n\n'
            'Serialized synopsis storage, model bytes and derived runtime-array costs are distinct. Resident Python '
            'query timings exclude retrieval, initialization and exact fallback. Logged SA scores are quality references '
            'only. No E2E speedup, new-data generalization or formal risk guarantee is established here.\n\n'
            'A useful prototype should retain quality/acceptance advantages over merged_only and same-budget finer '
            'histograms without relying on higher violation. Confirm on cell groups and fresh queries before expanding. '
            'Seeds and rates repeat queries; summary counts are descriptive cases, not independent statistical trials.\n')
        export_compact_results(destination)
        print('COMPLETE:',destination,flush=True)
        return destination,quality,summary
    except BaseException as error:
        state.update(status='failed',error=f'{type(error).__name__}: {error}',wall_seconds=time.perf_counter()-started)
        save_json(destination/'run.json',state);print('FAILED;diagnostics preserved:',destination,flush=True);raise

## Execute the prototype

This trains new models because the feature definitions have changed. The source-free provider uses only the serialized
histogram; compact providers additionally use the persisted map. Full-source data is available only to the explicit reference.
Resident feature extraction and one-query native inference are timed separately from construction, initialization and batch
evaluation. Live scores and frozen-gate decisions must match saved batch predictions. Timing sample selection is independent
of scores/truth. These microtimings exclude retrieval, routing and fallback.

In [ ]:
# RUN_PROTOTYPE_CELL
config=load_settings()
result_dir,quality_by_setting,summary=run_prototype(config)
display(summary) if 'display' in globals() else print(summary.to_string(index=False))

## Results to inspect

- `quality_by_setting.csv`: per-split/seed/base-rate acceptance, violation counts/rates, good-answer recall and AUC.
- `fixed_coverage.csv`: predeclared 5%/10% score-only ranking diagnostics, retaining complete ties; not deployment gates.
- `paired_deltas.csv`: compact versus merged-only and matching finer baseline. Compare violation and sample counts with gains.
- `budgets.csv`, `payloads.csv`, `metadata_build.csv`: actual bytes, finer choices/slack, numerical moment diagnostics.
- `resident_timing_summary.csv`, `payload_loads.csv`, `models.csv`: resident cost, local setup loads, fit time/trees/model bytes.
- `test_predictions.csv`, `splits.csv`, `estimate_parity.csv`, `input_manifest.json`, `run.json`: reproducibility and validation.

Send **`compact_metadata_results.zip`** after completion. It includes compact analysis CSVs/figures/provenance, not model
weights or histogram payloads. Model/payload artifacts remain in the result directory for reruns or deeper debugging.

Continue only if compact information improves the quality/bytes tradeoff over both merged-only and the within-budget
finer baseline without relying on higher violation. Confirm with cell groups and fresh queries before claiming generality.
If the signal is absent, stop this representation design rather than adding many models. Existing 09/09b remain separate.

In [ ]:
# PLOT_PROTOTYPE_CELL
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(12,4))
for (split,variant,score),group in quality_by_setting.groupby(['split_kind','variant','score']):
    label=f'{split}/{variant}/{score}'
    axes[0].scatter(group.acceptance*100,group.violation*100,label=label,alpha=.75)
    if score=='learned':
        payload=pd.read_csv(Path(result_dir)/'payloads.csv')
        sizes=[payload[(payload.variant==variant)&np.isclose(payload.base_rate,r)].total_synopsis_payload_bytes.iloc[0]/1024
               for r in group.base_rate]
        axes[1].scatter(sizes,group.good_recall*100,label=label,alpha=.75)
axes[0].axhline(5,color='black',linestyle='--')
axes[0].set(xlabel='Accepted queries (%)',ylabel='Violation among accepted (%)',title='Quality accompanies acceptance')
axes[1].set(xlabel='Serialized synopsis payload (KiB)',ylabel='Good-answer recall (%)',title='Quality versus actual stored bytes')
for axis in axes:
    axis.grid(alpha=.2)
axes[0].legend(fontsize=5,bbox_to_anchor=(0,-.2),loc='upper left',ncol=2)
fig.tight_layout();fig.savefig(Path(result_dir)/'quality_budget.png',dpi=160,bbox_inches='tight');plt.show()
export_compact_results(Path(result_dir))
print('Upload:',Path(result_dir)/'compact_metadata_results.zip')